# ezcal quick tour

`ezcal` (Easy Calculation) turns a bare crystal structure into a finished
first-principles calculation with one command.  This notebook shows the
Python API behind the CLI:

1. get a structure (local CIF or Materials Project)
2. look at what ezcal would do (symmetry, k-mesh, pseudopotentials, cutoffs)
3. run a full `relax -> scf -> nscf -> dos -> bands` workflow
4. draw the results with matplotlib and plotly
5. do the same thing with a machine-learning potential (SevenNet)

Kernel: `/home/tajimamainpc/.venv/ezcalenv312`

In [ ]:
import os, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import ezcal
from ezcal.config import load_config
from ezcal.structures import read_structure, standardize, structure_info, auto_kmesh, band_path
from ezcal.pseudo import PseudoManager
from ezcal.workflows import Workflow

print('ezcal', ezcal.__version__)
WORK = Path('..').resolve() / '02_ezcal_test' / 'notebook_runs'
WORK.mkdir(parents=True, exist_ok=True)
WORK

## 1. Materials Project API key

The key is typed here rather than stored in the repository.
Leave it empty to skip the download and use the local CIF files instead.

In [ ]:
import getpass

MP_API_KEY = os.environ.get('MP_API_KEY') or getpass.getpass('Materials Project API key (blank to skip): ')
os.environ['MP_API_KEY'] = MP_API_KEY
print('key set' if MP_API_KEY else 'no key - local structures will be used')

In [ ]:
if MP_API_KEY:
    structure = read_structure('mp-149', api_key=MP_API_KEY)   # Si
else:
    structure = read_structure('../02_ezcal_test/structures/Si.cif')

structure = standardize(structure, primitive=True)
structure

## 2. What will ezcal do?

Cutoffs come from the pseudopotential headers, the k-mesh from a
reciprocal-space spacing, and the band path from seekpath - nothing has to
be looked up by hand.

In [ ]:
info = structure_info(structure)
print(json.dumps(info, indent=2))
print('k-mesh @ kspacing=0.25 :', auto_kmesh(structure, 0.25))

pm = PseudoManager(functional='pbe')
pseudos = pm.resolve_all([str(el) for el in structure.composition.elements])
for element, p in pseudos.items():
    print(f'{element:3s} {p.filename}  type={p.pseudo_type} Z={p.z_valence:g} ecutwfc={p.ecutwfc} Ry')
print('suggested cutoffs:', PseudoManager.suggest_cutoffs(pseudos.values()))

In [ ]:
bp = band_path(structure)
print(bp.nkpt, 'k-points')
print(' -> '.join(dict.fromkeys(l for _, l in bp.labels if l)))

## 3. Run the whole workflow

`Workflow.run('auto')` chains `vc-relax -> scf -> nscf -> dos -> bands`,
reusing one charge density.  The settings below are deliberately loose so
that the notebook finishes in about a minute on 8 cores.

In [ ]:
cfg = load_config()
cfg.apply_overrides({
    'run.nproc': 8,
    'dft.ecutwfc': 30, 'dft.ecutrho': 240,
    'dft.kmesh': [6, 6, 6],
    'dft.conv_thr': 1e-6,
    'output.plot': ['both'],
})

wf = Workflow(cfg, structure, WORK / 'Si_auto', label='Si', log=print)
result = wf.run('auto')
print('ok:', result.ok)
for name in result.order:
    s = result.steps[name]
    print(f'{name:10s} E={s.energy} eV  E_F={s.fermi_energy} eV  gap={s.band_gap}')

In [ ]:
print(json.dumps(result.summary()['steps'], indent=2)[:2000])

## 4. Draw the results

The workflow already wrote PNG and interactive HTML into `plots/`.
The same helpers can be called directly to draw inside the notebook.

In [ ]:
from IPython.display import Image, display

for name in ['bands.png', 'dos.png', 'bands_dos.png']:
    path = wf.rundir / 'plots' / name
    if path.exists():
        display(Image(filename=str(path)))

In [ ]:
# interactive plotly version, rendered inline
from ezcal import plotting
import plotly.io as pio

bands, dos = result.steps.get('bands'), result.steps.get('dos')
fermi = (result.steps['bands'].data.get('gap_info') or {}).get('vbm')
arr = plotting.band_arrays(bands, fermi)

import plotly.graph_objects as go
fig = go.Figure()
eig, dist = arr['eigenvalues'], arr['distances']
for b in range(eig.shape[2]):
    fig.add_trace(go.Scatter(x=dist, y=eig[0, :, b] - arr['fermi'], mode='lines',
                             line=dict(color='#1f4e9c', width=1.3), showlegend=False))
ticks, names = plotting._tick_positions(dist, arr['labels'])
fig.update_layout(template='plotly_white', height=460, width=720,
                  yaxis=dict(title='E - E_VBM (eV)', range=[-10, 10]),
                  xaxis=dict(tickvals=ticks, ticktext=names))
fig.show()

## 5. The same structure with a machine-learning potential

SevenNet answers energy / forces / stress in a fraction of a second, which
makes it a good pre-relaxation step before the DFT run.  It cannot do
electronic structure, and ezcal says so rather than producing a wrong plot.

In [ ]:
mcfg = load_config()
mcfg.apply_overrides({'engine': 'mlip', 'mlip.model': '7net-0', 'mlip.device': 'cpu'})

mwf = Workflow(mcfg, structure, WORK / 'Si_mlip', label='Si (SevenNet)', log=print)
mres = mwf.run('vc-relax')
step = mres.steps['vc-relax']
print('E/atom  MLIP :', step.energy_per_atom)
print('E/atom  DFT  :', result.steps['scf'].energy_per_atom, '(different reference, do not compare directly)')
print('lattice MLIP :', step.structure.lattice.abc)
print('lattice DFT  :', result.structure_final.lattice.abc)

## 6. Submitting to a queue

`scheduler: qsub` renders `run_qe.sh` and submits it.  There is no queue on
this machine, so `run.dry_run` is used to look at the script that *would*
be submitted.

In [ ]:
qcfg = load_config()
qcfg.apply_overrides({'run.scheduler': 'qsub', 'run.dry_run': True, 'run.nproc': 16,
                      'run.qsub.queue': 'batch', 'run.qsub.walltime': '12:00:00'})
qwf = Workflow(qcfg, structure, WORK / 'Si_qsub', label='Si', log=print)
qres = qwf.run('scf')
print(open(qwf.rundir / '00_scf' / 'scf.qsub.sh').read())

## 7. Where everything ended up

Each run directory holds the inputs, the raw QE output, `summary.json`,
`report.md`, `raw_data.json` (for `ezcal plot`) and the figures.

In [ ]:
for path in sorted(wf.rundir.rglob('*')):
    if path.is_file() and path.suffix in {'.md', '.json', '.png', '.html', '.csv', '.in'}:
        print(path.relative_to(wf.rundir))